In [1]:
import asyncio

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore
from langgraph.types import Interrupt

from common.runner import GraphRunner
from usecases.research.deep_researcher import build_graph
from usecases.research.state import AgentState


def handle_interrupt(interrupt: Interrupt):
    msg = ""
    for m in interrupt.value:
        msg += input(m)
        msg += "\n"
    return msg

In [2]:
wf = build_graph().compile(
    checkpointer=InMemorySaver(),
    store=InMemoryStore(),
)
configuration = {
    "configurable": {
        "max_concurrent_research_units": 1,
        "max_researcher_iterations": 6,
        "thread_id": "1",
        "model_config": {
            "summarize_model": {
                "provider": "OpenAI",
                "model": "/data_new/wyl/antibody_gen/foundry/output/Qwen-8b-run-2/full_finetuned_model/",
                "params": {
                    "api_key": "FAKE",
                    "base_url": "http://127.0.0.1:9011/v1",
                    "temperature": 0.9,
                },
            },
            "reasoning_model": {
                "provider": "OpenAI",
                "model": "/data_new/wyl/antibody_gen/foundry/output/Qwen-8b-run-2/full_finetuned_model/",
                "params": {
                    "api_key": "FAKE",
                    "base_url": "http://127.0.0.1:9011/v1",
                    "temperature": 0.9,
                },
            },
        },
        "mcp_config": {"service_ids": []},
    }
}
init_state: AgentState = {
    "supervisor_messages": [],
    "research_brief": None,
    "final_report": "",
    "messages": [
        "Please do a research on development of anti-GABA or anti-GABA receptor antibodies for research and clinical"
    ],
}
runner = (
    GraphRunner(wf)
    .with_value_handler(lambda v: print(v))
    .with_message_handler(lambda v: print(v.content, end=""))
)


async def run():
    ret = await runner.run(init_state, configuration)
    print(ret)
    while ret is not None:
        ret = await runner.resume(handle_interrupt(ret), configuration)


print(wf.get_state(configuration))
await run()

StateSnapshot(values={}, next=(), config={'configurable': {'max_concurrent_research_units': 1, 'max_researcher_iterations': 6, 'thread_id': '1', 'model_config': {'summarize_model': {'provider': 'OpenAI', 'model': '/data_new/wyl/antibody_gen/foundry/output/Qwen-8b-run-2/full_finetuned_model/', 'params': {'api_key': 'FAKE', 'base_url': 'http://127.0.0.1:9011/v1', 'temperature': 0.9}}, 'reasoning_model': {'provider': 'OpenAI', 'model': '/data_new/wyl/antibody_gen/foundry/output/Qwen-8b-run-2/full_finetuned_model/', 'params': {'api_key': 'FAKE', 'base_url': 'http://127.0.0.1:9011/v1', 'temperature': 0.9}}}, 'mcp_config': {'service_ids': []}}}, metadata=None, created_at=None, parent_config=None, tasks=(), interrupts=())
{'messages': [HumanMessage(content='Please do a research on development of anti-GABA or anti-GABA receptor antibodies for research and clinical', additional_kwargs={}, response_metadata={}, id='d7e9227f-6b34-4f31-8a99-fac6ab85978f')], 'supervisor_messages': [], 'raw_notes': 

In [ ]:
state: StateSnapshot = wf.get_state(configuration)
print(len("\n".join(state.values["citations"])))

In [ ]:
from langgraph.types import StateSnapshot

from common.util.retrieval_utils import remove_think_tags
from usecases.research.deep_researcher import process_citation

state: StateSnapshot = wf.get_state(configuration)

print(state.values["final_report"])



[1] Typical clinical and imaging manifestations of encephalitis with anti-γ-aminobutyric acid B receptor antibodies: clinical experience and a literature review (https://pubmed.ncbi.nlm.nih.gov/32508739/)  
[2] The GABAergic system is a critical target for anaesthetics and analgesics (https://pmc.ncbi.nlm.nih.gov/articles/PMC9538268/)  
[3] as possible mechanisms contributing to seizure disorders (https://pmc.ncbi.nlm.nih.gov/articles/PMC10235571/)  
[4] entorhinal cortex. GABA B R antibodies do not appear to affect GABA B Rs by internalization but rather reduce excitability on the medial temporal lobe networks (https://pmc.ncbi.nlm.nih.gov/articles/PMC5862107/)  
[5] Antibodies against GAD are associated with various autoimmune and neurological conditions (https://pmc.ncbi.nlm.nih.gov/articles/PMC5107286/)  
[6] The GABA A receptor (GABA A R) is a ligand-gated chloride channel that mediates fast inhibitory synaptic transmission in the CNS (https://pmc.ncbi.nlm.nih.gov/articles/PMC53

In [ ]:
from common.factory import get_reasoning_model
from usecases.retrieval.tools import retrieve, retrieve_doc, web_search_node

configuration = {
    "configurable": {
        "max_concurrent_research_units": 1,
        "max_researcher_iterations": 6,
        "thread_id": "1",
        "model_config": {
            "summarize_model": {
                "provider": "OpenAI",
                "model": "/data_new/wyl/antibody_gen/foundry/output/Qwen-8b-run-2/full_finetuned_model/",
                "params": {
                    "api_key": "FAKE",
                    "base_url": "http://127.0.0.1:9011/v1",
                },
            },
            "reasoning_model": {
                "provider": "OpenAI",
                "model": "/data_new/wyl/antibody_gen/foundry/output/Qwen-8b-run-2/full_finetuned_model/",
                "params": {
                    "api_key": "FAKE",
                    "base_url": "http://127.0.0.1:9011/v1",
                },
            },
        },
        "mcp_config": {"service_ids": []},
    }
}

for c in get_reasoning_model(configuration).stream(
    "write a section in academic report, introducing anti-GABA receptor"
):
    print(c.content, end="")